In [ ]:
# ============================================================
# DATASET 1: PLAY TENNIS
# ============================================================

import pandas as pd
import numpy as np
from math import log2
from collections import Counter
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# -----------------------------
# PLAY TENNIS DATASET
# -----------------------------
data = pd.DataFrame({
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast','Sunny'],
    'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool','Mild','Cool'],
    'Humidity': ['High','High','High','High','Normal','Normal','Normal','High'],
    'Wind': ['Weak','Strong','Weak','Weak','Weak','Strong','Strong','Weak'],
    'PlayTennis': ['No','No','Yes','Yes','Yes','No','Yes','No']
})

X = data.drop("PlayTennis", axis=1)
y = data["PlayTennis"]
y_encoded = y.map({'No':0, 'Yes':1})

# -----------------------------
# ID3 IMPLEMENTATION
# -----------------------------

def entropy(y):
    counts = Counter(y)
    total = len(y)
    return -sum((count/total) * log2(count/total) for count in counts.values())

def information_gain(X, y, feature):
    total_entropy = entropy(y)
    values = X[feature].unique()
    weighted_entropy = 0
    for v in values:
        subset = y[X[feature] == v]
        weighted_entropy += (len(subset)/len(y)) * entropy(subset)
    return total_entropy - weighted_entropy

def id3(X, y, features):
    if len(set(y)) == 1:
        return y.iloc[0]
    if not features:
        return y.mode()[0]

    gains = [information_gain(X, y, f) for f in features]
    best_feature = features[np.argmax(gains)]

    tree = {best_feature: {}}
    for value in X[best_feature].unique():
        subset_X = X[X[best_feature] == value]
        subset_y = y[X[best_feature] == value]
        subtree = id3(subset_X, subset_y,
                      [f for f in features if f != best_feature])
        tree[best_feature][value] = subtree
    return tree

def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree
    feature = next(iter(tree))
    value = sample[feature]
    return predict(tree[feature][value], sample)

print("\n========== PLAY TENNIS - ID3 ==========")

tree = id3(X, y, list(X.columns))
y_pred_id3 = X.apply(lambda row: predict(tree, row), axis=1)
y_pred_id3 = y_pred_id3.map({'No':0, 'Yes':1})

accuracy = accuracy_score(y_encoded, y_pred_id3)
print("Accuracy:", accuracy)
print("Precision:", precision_score(y_encoded, y_pred_id3))
print("Recall:", recall_score(y_encoded, y_pred_id3))
print("F1 Score:", f1_score(y_encoded, y_pred_id3))
print("Confusion Matrix:\n", confusion_matrix(y_encoded, y_pred_id3))
print("Error:", 1 - accuracy)

# -----------------------------
# CART IMPLEMENTATION
# -----------------------------

print("\n========== PLAY TENNIS - CART ==========")

le = LabelEncoder()
X_encoded = X.apply(le.fit_transform)

cart_model = DecisionTreeClassifier(criterion="gini", random_state=0)
cart_model.fit(X_encoded, y_encoded)
y_pred_cart = cart_model.predict(X_encoded)

accuracy = accuracy_score(y_encoded, y_pred_cart)
print("Accuracy:", accuracy)
print("Precision:", precision_score(y_encoded, y_pred_cart))
print("Recall:", recall_score(y_encoded, y_pred_cart))
print("F1 Score:", f1_score(y_encoded, y_pred_cart))
print("Confusion Matrix:\n", confusion_matrix(y_encoded, y_pred_cart))
print("Error:", 1 - accuracy)


# ============================================================
# DATASET 2: IRIS DATASET
# ============================================================

print("\n\n================================================")
print("DATASET 2: IRIS")
print("================================================")

iris = load_iris()

X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42)

# -----------------------------
# ID3 (Entropy)
# -----------------------------

print("\n========== IRIS - ID3 ==========")

id3_model = DecisionTreeClassifier(criterion="entropy", random_state=42)
id3_model.fit(X_train, y_train)
y_pred_id3 = id3_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred_id3)
print("Accuracy:", accuracy)
print("Precision:", precision_score(y_test, y_pred_id3, average='macro'))
print("Recall:", recall_score(y_test, y_pred_id3, average='macro'))
print("F1 Score:", f1_score(y_test, y_pred_id3, average='macro'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_id3))
print("Error:", 1 - accuracy)

# -----------------------------
# CART (Gini)
# -----------------------------

print("\n========== IRIS - CART ==========")

cart_model = DecisionTreeClassifier(criterion="gini", random_state=42)
cart_model.fit(X_train, y_train)
y_pred_cart = cart_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred_cart)
print("Accuracy:", accuracy)
print("Precision:", precision_score(y_test, y_pred_cart, average='macro'))
print("Recall:", recall_score(y_test, y_pred_cart, average='macro'))
print("F1 Score:", f1_score(y_test, y_pred_cart, average='macro'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_cart))
print("Error:", 1 - accuracy)


========== PLAY TENNIS - ID3 ==========
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Confusion Matrix:
 [[4 0]
 [0 4]]
Error: 0.0

========== PLAY TENNIS - CART ==========
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Confusion Matrix:
 [[4 0]
 [0 4]]
Error: 0.0


DATASET 2: IRIS

========== IRIS - ID3 ==========
Accuracy: 0.9777777777777777
Precision: 0.9761904761904763
Recall: 0.9743589743589745
F1 Score: 0.974320987654321
Confusion Matrix:
 [[19  0  0]
 [ 0 13  0]
 [ 0  1 12]]
Error: 0.022222222222222254

========== IRIS - CART ==========
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Confusion Matrix:
 [[19  0  0]
 [ 0 13  0]
 [ 0  0 13]]
Error: 0.0
